# Tutorial 9: Does a random forest make predictions more stable?

## Briefing for the responsible study group

| Planning item | Allocation |
|---|---|
| Presentation date | Tuesday 1 December 2026 |
| Prerequisite lecture | Tree-Based Algorithms, 23 November 2026 |
| Preparation time after prerequisite | 5 full weekdays, excluding lecture and tutorial dates |
| Format | 15-minute presentation + approximately 15-minute discussion |

**Lecture connection:** Tree-Based Algorithms slides 34–37 and 43–52: complexity, pruning, ensemble averaging, bootstrap sampling and random subspaces.

This is your group's working notebook. It replaces the requirement to augment a section of an existing course notebook. The supplied cells establish a starting point; your group develops and teaches the investigation.

### Your teaching challenge

The lecture explains that averaging trees can reduce variance. Make that argument visible by repeatedly fitting models to slightly different training samples and comparing how much their predictions change. Distinguish stability from predictive accuracy: a consistently wrong model can be very stable.

Use the supplied synthetic regression problem with one informative feature and three irrelevant features. Compare a single decision tree with a random forest. The helper generates independent training samples of the same size and an independent fixed test set. A fixed prediction grid varies the informative feature while holding the others at zero; this is a teaching slice, not the entire input space.

### Core investigation

1. Fit a tree and forest to each of a modest number of independently drawn training samples, keeping their hyperparameters fixed. Use paired samples: both methods see exactly the same data in each repeat.
2. Plot their prediction curves on the fixed grid. Summarize variation across repeats—for example, pointwise standard deviation and a band around the mean prediction. Label the band as variation across fitted models, not a confidence interval or prediction interval.
3. Compare held-out RMSE using the same fixed test observations. Show accuracy and stability separately. Use enough repeats to reveal the pattern without making the notebook slow; about 20 repeats and 50 trees per forest is sufficient as a starting point.
4. Explain the roles of bootstrap sampling, random feature subsets and averaging. Explain why correlated trees limit the benefit and why stability alone does not prove that bias is low.
5. Include a shallow, deliberately restricted tree as a small contrasting condition. Explain how restricting complexity can reduce variation while limiting approximation of the true signal. Compare its held-out RMSE with both the flexible tree and the forest; a shallower tree need not have worse predictive error than a deeper tree.

### What classmates should gain

They should connect the abstract variance argument to repeated fitted predictions, and understand that a forest's individual trees can be highly flexible even when their average is comparatively stable. Make clear whether variation in your experiment comes from changing training data, algorithmic randomness or both.

### Boundaries and support

Data generation, model factories, a repeated-fit loop and summary helpers are supplied. Your contribution is to use these tools to create a clear experiment and explanation. Keep model random seeds fixed across repeats in the core experiment to isolate sensitivity to training samples. Do not add boosting, feature importance or extensive tuning. **Optional:** vary the number of trees while keeping everything else fixed, or discuss feature subsampling in the presence of irrelevant features.

## How to turn your investigation into a tutorial

Your group has **15 minutes to present**, followed by approximately **15 minutes of discussion** with the instructor and class. Your audience has attended the prerequisite lecture but has not done your investigation. Help them understand one important question through a focused, reproducible demonstration.

**Required contribution.** Complete the core investigation below, explain why the results occur, and discuss at least one limitation or contrasting condition. Select two or three main figures or tables that make your argument clear. A useful tutorial connects the question, experimental design, evidence and conclusion; a sequence of unexplained code cells is insufficient. You are not expected to build a production system, implement algorithms from scratch, collect external data or run an extensive model search.

**What to submit.** Submit your completed `.ipynb` notebook with explanations and reproducible code, plus any slides or other presentation material you use. Follow the instructor's announced submission deadline. Keep the supplied briefing, but edit, reorganize or replace the working sections as needed. Identify substantial changes to supplied helpers. Record package versions, seeds and relevant settings; restart the kernel and run all cells before submission. Clearly distinguish supplied material from your group's contribution.

**Presentation format is your choice.** You may present from the notebook, create slides, render to HTML, develop an interactive demonstration or combine these. Make the selected format work within 15 minutes and have a fallback for live demonstrations. Creativity earns credit when it improves understanding, engagement or the quality of the investigation. Technical spectacle should be avoided. A clear static plot can be excellent.

**AI assistance is allowed.** You may use AI to help develop code, explanations and visualizations. Your group remains responsible for correctness and must be able to explain the submitted work. Include a short account of the tools used, their main contributions, and how you checked those contributions. Verify at least one important result independently—for example, with a hand calculation, an analytic special case or a comparison with a trusted library function. A long prompt transcript is not required. Using AI is optional and is not itself rewarded.

**Assessment.** This component contributes **15% of the course grade**. Within the tutorial, the criteria are **conceptual understanding (40%)**, **added value for classmates (40%)**, and **quality of communication (20%)**. Added value comes from the investigation and how it develops understanding beyond the lecture and the supplied starter cells. Optional extensions are not required for an excellent grade. All groups should prioritize a sound core investigation over breadth.

**Suggested presentation rhythm:** about 3 minutes for the question and essential concepts, 8 minutes for the demonstration and evidence, and 4 minutes for interpretation and limitations. Prepare one question that invites the class to reason about your findings. Anticipate questions about experimental controls, assumptions and alternative explanations.

## Preliminaries: environment

The course environment provides all libraries needed to execute the starter code (see the [README for installation instructions](https://github.com/stefanlessmann/ESMT_IML)). You can install additional libraries if needed.

Run the supplied cells first. They provide a starting point, **not a completed tutorial**. Code cells marked `YOUR GROUP'S WORK` are intentionally unfinished and can be replaced or expanded.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display

SEED = 888
plt.rcParams.update({'figure.figsize': (8, 4.5), 'axes.grid': True,
                     'grid.alpha': 0.2, 'font.size': 11})
print(f'Python {sys.version.split()[0]} | NumPy {np.__version__} | scikit-learn {sklearn.__version__}')

## Preliminaries: supplied data and helpers

The supplied forest uses bootstrap=True and max_features=0.75. With four features, this considers three candidates at each split. Both core estimators can grow flexible trees, subject to a minimum leaf size. The shallow-tree contrast is supplied but not evaluated. Held-out target noise creates irreducible error; the known noiseless curve is only a teaching reference.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

def true_signal(x):
    return np.sin(2 * np.asarray(x)) + 0.25 * np.asarray(x)

def sample_regression(n, seed):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-3, 3, size=(n, 4))
    y = true_signal(X[:, 0]) + rng.normal(0, 0.35, n)
    return X, y

X_test, y_test = sample_regression(1500, seed=9000)
grid_x = np.linspace(-3, 3, 240)
X_grid = np.column_stack([grid_x, np.zeros((len(grid_x), 3))])
training_seeds = list(range(100, 120))

def tree_factory():
    return DecisionTreeRegressor(min_samples_leaf=2, random_state=SEED)

def forest_factory():
    return RandomForestRegressor(n_estimators=50, min_samples_leaf=2,
                                 max_features=0.75, bootstrap=True,
                                 random_state=SEED, n_jobs=1)

def shallow_tree_factory():
    return DecisionTreeRegressor(max_depth=2, min_samples_leaf=2, random_state=SEED)

def run_repeated_fits(factory, seeds=training_seeds, n_train=180):
    grid_predictions, test_rmse = [], []
    for seed in seeds:
        X_train, y_train = sample_regression(n_train, seed)
        model = factory()
        model.fit(X_train, y_train)
        grid_predictions.append(model.predict(X_grid))
        test_rmse.append(np.sqrt(mean_squared_error(y_test, model.predict(X_test))))
    return {'grid_predictions': np.asarray(grid_predictions),
            'test_rmse': np.asarray(test_rmse)}

def summarize_repeats(result):
    predictions = result['grid_predictions']
    return {'mean_test_rmse': float(result['test_rmse'].mean()),
            'sd_test_rmse': float(result['test_rmse'].std(ddof=1)),
            'mean_pointwise_prediction_sd': float(predictions.std(axis=0, ddof=1).mean())}

print('Prepared:', len(training_seeds), 'paired training samples; fixed test cases:', len(y_test))
# Call run_repeated_fits(...) in your own experiment cells below.

## Investigation 1: Run the paired stability experiment

Call the helper for the tree and forest with identical training_seeds. Explain what varies across repeats and why the same fixed test set/grid is used.

Add your explanation, code and evidence below.

In [ ]:
# YOUR GROUP'S WORK
# Add your investigation here. Split this cell into several cells if helpful.

*Your interpretation of the results: replace this prompt with an explanation.*

## Investigation 2: Make stability and accuracy visible

Plot the prediction curves or mean curves with clearly labelled model-variation bands. Summarize grid prediction variability and test RMSE. Check the helper with one direct model fit.

Add your explanation, code and evidence below.

In [ ]:
# YOUR GROUP'S WORK
# Add your investigation here. Split this cell into several cells if helpful.

*Your interpretation of the results: replace this prompt with an explanation.*

## Investigation 3: Add the contrasting condition

Evaluate the supplied shallow tree. Use the known noiseless curve as a teaching reference to discuss systematic approximation error separately from variation across fits and held-out RMSE. A shallow tree can predict better than an overfitting deep tree even while approximating the signal less flexibly; report the observed comparison rather than assuming an ordering. Connect the findings to averaging, tree correlation and the role of irrelevant features.

Add your explanation, code and evidence below.

In [ ]:
# YOUR GROUP'S WORK
# Add your investigation here. Split this cell into several cells if helpful.

*Your interpretation of the results: replace this prompt with an explanation.*

## Your teaching conclusions

Replace this text with:

1. A direct answer to the tutorial's central question, supported by your results.
2. Two or three takeaways that classmates should remember.
3. One limitation, counterexample or condition under which the result changes.
4. One discussion question, plus notes on how your group would answer it.

## Contribution and verification record

- **Supplied material retained or changed:** …
- **Your group's experimental and explanatory contributions:** …
- **Independent check of an important result:** …
- **AI tools and their contributions, if any:** …
- **How generated code/explanations were checked:** …
- **Package versions, random seeds and reproduction instructions:** …
- **Group members and contributions:** …

## Before presenting

- [ ] Restarted the kernel and ran all cells successfully.
- [ ] Checked that splits, metrics, units and class labels are correct.
- [ ] Distinguished evidence from interpretation; avoided claims stronger than the experiment supports.
- [ ] Labelled plots and made figures legible for classmates.
- [ ] Rehearsed within 15 minutes and prepared for discussion.
- [ ] Included the completed notebook and any additional presentation material.